# Bayesian Negative Binomial — Global Hierarchical Model

**Prerequisites:** Run `loo/LOO.ipynb` at least once to generate
`outputs/scores/selected_covariates.json`, or set `SELECTED_COVARIATES`
manually in the Config cell below.

## 1. Environment

In [ ]:
import os
os.environ['JAX_PLATFORMS'] = 'cuda'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

import jax
print('JAX devices:', jax.devices())

In [ ]:
import json
import warnings
import pandas as pd
import pymc as pm
import arviz as az

from utils    import add_season_labels, epiweek_to_date, Region_states
from train    import run_validation, run_forecast
from plotting import plot_forecast_grid, plot_scores_heatmap

warnings.filterwarnings('ignore')
os.makedirs('outputs/plots',       exist_ok=True)
os.makedirs('outputs/scores',      exist_ok=True)
os.makedirs('outputs/predictions', exist_ok=True)

print(f'PyMC  : {pm.__version__}')
print(f'ArviZ : {az.__version__}')

## 2. Config

In [ ]:
VAL_SEASONS     = ['2022-2023', '2023-2024', '2024-2025']
FORECAST_SEASON = '2025-2026'

# MCMC settings
DRAWS         = 1000
TUNE          = 1000
CHAINS        = 4
TARGET_ACCEPT = 0.90
SEED          = 42

TEST_MODE = False
if TEST_MODE:
    DRAWS   = 500
    TUNE    = 500
    CHAINS  = 2
    ACTIVE_STATES = ['SP', 'MG', 'CE']
else:
    ACTIVE_STATES = sorted([s for states in Region_states.values() for s in states])

print(f'TEST_MODE: {TEST_MODE}')
print(f'States ({len(ACTIVE_STATES)}): {ACTIVE_STATES}')
print(f'DRAWS={DRAWS}  TUNE={TUNE}  CHAINS={CHAINS}')

## 3. Load Data

In [ ]:
df           = pd.read_csv('data/Hist_dataset.csv.gz',    compression='gzip')
df_fcst_clim = pd.read_csv('data/forecast_dataset.csv.gz', compression='gzip')

df           = add_season_labels(df)
df_fcst_clim = add_season_labels(df_fcst_clim)

df           = df[df['uf'].isin(ACTIVE_STATES)].copy()
df_fcst_clim = df_fcst_clim[df_fcst_clim['uf'].isin(ACTIVE_STATES)].copy()

print(f'Historical  : {df.shape[0]:,} rows')
print(f'Forecast    : {df_fcst_clim.shape[0]:,} rows')
print(f'Seasons     : {sorted(df["season"].unique())}')

## 4. Load LOO-Selected Covariates

Loads from `outputs/scores/selected_covariates.json` (produced by `loo/LOO.ipynb`).
Override `SELECTED_COVARIATES` manually if you want to test a specific set.

In [ ]:
LOO_RESULT_PATH = 'outputs/scores/selected_covariates.json'

if os.path.exists(LOO_RESULT_PATH):
    with open(LOO_RESULT_PATH) as f:
        loo_data = json.load(f)
    SELECTED_COVARIATES = loo_data['selected']
    print(f'Loaded from {LOO_RESULT_PATH}')
else:
    # Fallback — manually specify covariates if LOO has not been run
    print(f'WARNING: {LOO_RESULT_PATH} not found. Using fallback covariate list.')
    SELECTED_COVARIATES = [
        'sin_ew', 'cos_ew',
        'temp_med_lag_0', 'rel_humid_med_lag_0',
        'temp_med_lag_8', 'rel_humid_med_lag_8', 'precip_med_lag_8',
        'enso', 'iod', 'pdo',
    ]

# Filter to columns present in the loaded data
SELECTED_COVARIATES = [c for c in SELECTED_COVARIATES if c in df.columns]
print(f'Selected covariates ({len(SELECTED_COVARIATES)}): {SELECTED_COVARIATES}')

## 5. Validation

In [ ]:
all_scores, val_results = run_validation(
    df            = df,
    val_seasons   = VAL_SEASONS,
    active_states = ACTIVE_STATES,
    selected_covs = SELECTED_COVARIATES,
    draws         = DRAWS,
    tune          = TUNE,
    chains        = CHAINS,
    target_accept = TARGET_ACCEPT,
    seed          = SEED,
)

## 6. Forecast 2025-2026

In [ ]:
all_forecasts, forecast_results = run_forecast(
    df              = df,
    df_fcst_clim    = df_fcst_clim,
    forecast_season = FORECAST_SEASON,
    active_states   = ACTIVE_STATES,
    selected_covs   = SELECTED_COVARIATES,
    draws           = DRAWS,
    tune            = TUNE,
    chains          = CHAINS,
    target_accept   = TARGET_ACCEPT,
    seed            = SEED,
)

## 7. Plots

In [ ]:
for season in VAL_SEASONS:
    if val_results[season]:
        plot_forecast_grid(
            val_results[season], season, mode='validation',
            save_path=f'outputs/plots/Validation_{season}.png',
        )

if forecast_results:
    plot_forecast_grid(
        forecast_results, FORECAST_SEASON, mode='forecast',
        save_path=f'outputs/plots/Forecast_{FORECAST_SEASON}.png',
    )

## 8. Scores Summary

In [ ]:
import pandas as pd

scores_df = pd.DataFrame(all_scores)

print('=== MEAN BY SEASON ===')
print(scores_df.groupby('season')[['WIS','CRPS','LogS','MAE','RMSE']]
      .mean().round(2).to_string())

print('\n=== MEAN BY STATE ===')
print(scores_df.groupby('uf')[['WIS','CRPS','LogS','MAE','RMSE']]
      .mean().round(2).to_string())

plot_scores_heatmap(
    scores_df,
    save_path='outputs/plots/WIS_Heatmap.png',
)

## 9. Save Outputs

In [ ]:
# Validation scores
scores_df.to_csv('outputs/scores/validation_scores.csv', index=False)
print('Saved: outputs/scores/validation_scores.csv')

# Validation predictions
PRED_COLS = ['uf', 'season', 'pred', 'lower_50', 'upper_50',
             'lower_80', 'upper_80', 'lower_90', 'upper_90',
             'lower_95', 'upper_95', 'epiweek', 'date']

for season in VAL_SEASONS:
    if not val_results[season]:
        continue
    rows = []
    for res in val_results[season]:
        pi = res['pi_df'].rename(columns={'median': 'pred'}).copy()
        pi['uf']      = res['uf']
        pi['season']  = season
        pi['epiweek'] = res['df_season']['epiweek'].values
        pi['date']    = pi['epiweek'].apply(epiweek_to_date)
        rows.append(pi)
    out = pd.concat(rows).sort_values(['uf', 'date']).reset_index(drop=True)
    out[[c for c in PRED_COLS if c in out.columns]].to_csv(
        f'outputs/predictions/validation_{season}.csv', index=False
    )
    print(f'Saved: outputs/predictions/validation_{season}.csv')

# Forecast predictions
if all_forecasts:
    fc_df = pd.concat(all_forecasts).rename(columns={'median': 'pred'}).reset_index(drop=True)
    fc_df['date'] = fc_df['epiweek'].apply(epiweek_to_date)
    fc_df[[c for c in PRED_COLS if c in fc_df.columns]].sort_values(
        ['uf', 'date']
    ).reset_index(drop=True).to_csv(
        f'outputs/predictions/forecast_{FORECAST_SEASON}.csv', index=False
    )
    print(f'Saved: outputs/predictions/forecast_{FORECAST_SEASON}.csv')

print('Done.')